# Long-context vs RAG: the cost/accuracy tradeoff, measured

Million-token context windows tempt you to skip retrieval and paste the whole corpus into every
prompt. This notebook makes the tradeoff **concrete and measured** instead of a vibe:

- **Cost** — stuffing pays for the *whole corpus on every query*; RAG pays a small *fixed* amount.
  We find the **crossover** corpus size where RAG wins, with real token arithmetic.
- **Context dilution** — a real encoder proxy for lost-in-the-middle: more irrelevant context
  erodes the gold passage's retrieval lead. RAG keeps it wide.
- **Lost-in-the-middle** — the LLM accuracy-vs-position U-curve, **cited** from Liu et al. 2023.

> **Honesty (say it up front).** The **cost arithmetic** and the **encoder dilution proxy** are
> **our measurements** — real, reproducible, asserted here. The **lost-in-the-middle U-curve**
> (Liu et al. 2023), the **effective-vs-advertised context** gap (RULER), and **provider window
> sizes / prices** are **cited external** constants from their papers/docs — we never pass them
> off as our own measurement. Flagged again at each step.

> **Step 1 — import the canonical code + print the reproducibility banner.** Everything comes
> from `long_context_vs_rag.py` next to this notebook — the *same* module the page and figures
> use, which reuses chapter 5's real `DenseRetriever`. The banner prints numpy/torch **versions**
> and the detected **accelerator** (the encoder is CPU-pinned for determinism).

In [1]:
import numpy as np
import torch

from long_context_vs_rag import (
    DEFAULT_PRICE_PER_MTOK,
    LIU_U_CURVE,
    PROVIDERS,
    RETRIEVE_K,
    TOKENS_PER_CHUNK,
    DenseRetriever,
    cost_crossover_chunks,
    full_corpus,
    measure_dilution,
    query_cost_usd,
    rag_tokens,
    stuff_tokens,
)

accelerator = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print('numpy:', np.__version__)
print('torch:', torch.__version__, '| accelerator available:', accelerator, '| encoder runs on: cpu')

corpus = full_corpus()
dense = DenseRetriever(corpus)
print('corpus passages :', len(corpus))
print('dense lens      :', dense.backend)
print('chunk size (tok):', TOKENS_PER_CHUNK, '| RAG retrieves k =', RETRIEVE_K)

numpy: 2.4.6
torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


corpus passages : 11
dense lens      : sentence-transformers/all-MiniLM-L6-v2
chunk size (tok): 100 | RAG retrieves k = 5


## 1) The cost crossover (our measurement)

Stuffing the whole corpus costs `corpus_chunks × tokens_per_chunk` input tokens **every query**.
RAG retrieves only the top-k, so it costs a **fixed** `k × tokens_per_chunk + overhead` tokens no
matter how big the corpus is. The price cancels (both scale with the same $/token), so the
crossover is a pure token comparison.

> **Step 2 — compute both costs across corpus sizes and find the crossover.** RAG's per-query
> tokens are flat; stuffing's grow linearly with the corpus. The crossover is the smallest corpus
> where stuffing first exceeds RAG — beyond it, RAG is cheaper on *every* query.

In [2]:
rag_toks = rag_tokens()
crossover = cost_crossover_chunks()
price = DEFAULT_PRICE_PER_MTOK
print(f'RAG fixed cost: {rag_toks} tokens/query | price ${price:.2f}/1M tokens (representative, cited)')
print()
print(f"{'corpus (chunks)':>16} | {'stuff tokens':>12} | {'stuff $/q':>10} | {'RAG $/q':>9} | cheaper")
print('-' * 78)
# numeric order, with the just-below-crossover and crossover rows ADJACENT so the boundary is clear
for chunks in (crossover - 1, crossover, 10, 100, 1_000, 100_000):
    st = stuff_tokens(chunks)
    st_cost = query_cost_usd(st, price)
    rag_cost = query_cost_usd(rag_toks, price)
    cheaper = 'RAG' if rag_cost < st_cost else 'stuff'
    if chunks == crossover - 1:
        tag = '  <- just below (stuffing still wins)'
    elif chunks == crossover:
        tag = '  <- crossover (RAG wins from here)'
    else:
        tag = ''
    print(f'{chunks:>16,} | {st:>12,} | {st_cost:>9.5f}$ | {rag_cost:>8.5f}$ | {cheaper}{tag}')
print(f'\ncrossover corpus size: {crossover} chunks (~{crossover * TOKENS_PER_CHUNK:,} tokens)')

RAG fixed cost: 700 tokens/query | price $3.00/1M tokens (representative, cited)

 corpus (chunks) | stuff tokens |  stuff $/q |   RAG $/q | cheaper
------------------------------------------------------------------------------
               7 |          700 |   0.00210$ |  0.00210$ | stuff  <- just below (stuffing still wins)
               8 |          800 |   0.00240$ |  0.00210$ | RAG  <- crossover (RAG wins from here)
              10 |        1,000 |   0.00300$ |  0.00210$ | RAG
             100 |       10,000 |   0.03000$ |  0.00210$ | RAG
           1,000 |      100,000 |   0.30000$ |  0.00210$ | RAG
         100,000 |   10,000,000 |  30.00000$ |  0.00210$ | RAG

crossover corpus size: 8 chunks (~800 tokens)


> **Step 3 — assert the crossover before believing it.** At the crossover, stuffing first exceeds
> RAG; one chunk smaller, stuffing is still cheaper-or-equal. And at scale the gap explodes.

In [3]:
assert stuff_tokens(crossover) > rag_toks, 'at the crossover, stuffing must exceed RAG'
assert stuff_tokens(crossover - 1) <= rag_toks, 'just below, stuffing is still <= RAG'
factor = stuff_tokens(100_000) / rag_toks
print(f'at 100,000 chunks (~{stuff_tokens(100_000):,} tokens), stuffing costs {factor:,.0f}x RAG per query')
assert factor > 100, 'at 100k chunks stuffing should cost >100x RAG'
print('-> beyond a tiny corpus, RAG is decisively cheaper, and the gap only widens. All asserts passed.')

at 100,000 chunks (~10,000,000 tokens), stuffing costs 14,286x RAG per query
-> beyond a tiny corpus, RAG is decisively cheaper, and the gap only widens. All asserts passed.


## 2) Context dilution — an encoder proxy for lost-in-the-middle (our measurement)

The true lost-in-the-middle effect (an LLM attending worst to the *middle* of a long context)
needs a generative LLM — this env is encoder-only. So we measure a **retrieval-visible proxy**:
the gold passage's cosine **margin** over the best distractor, as the surrounding context grows.

> **Step 4 — measure the gold's margin shrinking as distractors pile up.** The gold's own cosine
> to the query is fixed (its text doesn't change). What changes is the *best distractor*: with
> more distractors present, one gets luckier and scores closer to the gold, so the gold's lead
> shrinks. RAG's focused top-k (≈ no distractors) keeps the margin wide.

In [4]:
query = 'When was the Helios-7 satellite launched?'
gold = corpus[0]
points = measure_dilution(dense, query, gold, (0, 5, 20, 100, 500))
print('query:', query)
print('gold :', gold)
print(f"gold's own cosine (fixed): {points[0].gold_cosine:.3f}\n")
print(f"{'# distractors':>14} | {'best distractor cos':>19} | {'gold margin':>11}")
print('-' * 52)
for p in points:
    print(f'{p.n_distractors:>14,} | {p.best_distractor_cosine:>19.3f} | {p.margin:>11.3f}')
focused, diluted = points[0].margin, points[-1].margin
print(f'\nfocused (RAG, 0 distractors) margin: {focused:.3f}')
print(f'diluted (500 distractors)    margin: {diluted:.3f}  ({(1 - diluted/focused)*100:.0f}% narrower)')
assert diluted < focused, 'more distractors must shrink the gold margin'
print('-> more irrelevant context erodes the gold lead (our measurement); RAG keeps it wide.')
print('   NOTE: this is a RETRIEVAL proxy for dilution, NOT the LLM mid-context drop (cited below).')

query: When was the Helios-7 satellite launched?
gold : The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
gold's own cosine (fixed): 0.804

 # distractors | best distractor cos | gold margin
----------------------------------------------------
             0 |               0.000 |       0.804
             5 |               0.600 |       0.204
            20 |               0.608 |       0.196
           100 |               0.615 |       0.189
           500 |               0.617 |       0.187

focused (RAG, 0 distractors) margin: 0.804
diluted (500 distractors)    margin: 0.187  (77% narrower)
-> more irrelevant context erodes the gold lead (our measurement); RAG keeps it wide.
   NOTE: this is a RETRIEVAL proxy for dilution, NOT the LLM mid-context drop (cited below).


## 3) Lost-in-the-middle — cited external (Liu et al. 2023, NOT our measurement)

The canonical evidence: LLM answer accuracy is highest when the relevant document is at the
**start or end** of the context and dips when it's in the **middle**. These are **Liu et al.'s
reported numbers**, not ours — we cite them to explain *why* retrieval (which places a few chunks
where the model reads best) beats burying the answer mid-prompt.

> **Step 5 — read Liu et al.'s reported U-curve.** This is external data (arXiv:2307.03172). We
> only assert the *shape* they reported — the middle is the worst position — to make the teaching
> point; we do not claim to have measured these accuracies.

In [5]:
print('CITED EXTERNAL: Liu et al. 2023, arXiv:2307.03172 (their reported result, not ours)')
print(f"{'gold position (of 20)':>21} | {'reported accuracy %':>19}")
print('-' * 44)
accs = [a for _, a in LIU_U_CURVE]
worst = min(accs)
for pos, acc in LIU_U_CURVE:
    marker = '  <- middle (worst)' if acc == worst else ''
    print(f'{pos:>21} | {acc:>18.1f}%{marker}')
first, mid = LIU_U_CURVE[0][1], worst
print(f'\ntheir reported edge -> middle drop: {first:.0f}% -> {mid:.0f}% (~{first-mid:.0f} points)')
# assert only the SHAPE they reported (middle worst), which is the teaching point
assert LIU_U_CURVE[2][1] == worst, 'Liu et al. report the MIDDLE position as the worst (the U dip)'
assert LIU_U_CURVE[0][1] > worst and LIU_U_CURVE[-1][1] > worst, 'edges beat the middle (the U shape)'
print('-> position matters: retrieve the few relevant chunks and place them where the model reads best.')

CITED EXTERNAL: Liu et al. 2023, arXiv:2307.03172 (their reported result, not ours)
gold position (of 20) | reported accuracy %
--------------------------------------------
                    1 |               75.0%
                    5 |               62.0%
                   10 |               54.0%  <- middle (worst)
                   15 |               60.0%
                   20 |               74.0%

their reported edge -> middle drop: 75% -> 54% (~21 points)
-> position matters: retrieve the few relevant chunks and place them where the model reads best.


## 4) The decision: stuff / retrieve / hybrid

Provider context windows are large but finite, and stuffing pays per token per query. The
decision comes down to corpus-size-vs-window and whether cost/latency matter.

> **Step 6 — the providers and the rule of thumb.** The window sizes are cited from provider docs
> (GPT-4o 128k, Claude 200k, Gemini 1.5 Pro 1M–2M). Even the largest window is finite, so a large
> enough corpus overflows *any* of them — which is when retrieval is the only option that scales.

In [6]:
for prov in PROVIDERS:
    window_chunks = prov.context_window // TOKENS_PER_CHUNK
    print(f'{prov.name:<24} window {prov.context_window:>9,} tok (~{window_chunks:,} chunks) '
          f'@ ${prov.price_per_mtok:.2f}/1M  [{prov.source}]')
print()
print('rule of thumb:')
print('  - corpus << window AND cost irrelevant  -> STUFF it (simplest)')
print(f'  - corpus >> window OR cost/latency matter -> RETRIEVE (RAG; the {crossover}-chunk crossover)')
print('  - need many relevant chunks              -> HYBRID (retrieve many -> long window)')
biggest = max(p.context_window for p in PROVIDERS)
assert stuff_tokens(1_000_000) > biggest, 'a 1M-chunk corpus overflows even a 2M-token window'
print(f'\n-> even the largest window ({biggest:,} tok) overflows at {biggest//TOKENS_PER_CHUNK:,}+ chunks:')
print('   corpora outgrow ANY window, so retrieval scales when stuffing cannot.')

GPT-4o (128k)            window   128,000 tok (~1,280 chunks) @ $2.50/1M  [OpenAI API pricing]
Claude Sonnet (200k)     window   200,000 tok (~2,000 chunks) @ $3.00/1M  [Anthropic pricing]
Gemini 1.5 Pro (1M–2M)   window 2,000,000 tok (~20,000 chunks) @ $1.25/1M  [Google Gemini pricing]

rule of thumb:
  - corpus << window AND cost irrelevant  -> STUFF it (simplest)
  - corpus >> window OR cost/latency matter -> RETRIEVE (RAG; the 8-chunk crossover)
  - need many relevant chunks              -> HYBRID (retrieve many -> long window)

-> even the largest window (2,000,000 tok) overflows at 20,000+ chunks:
   corpora outgrow ANY window, so retrieval scales when stuffing cannot.


## Try it yourself

Before you run the next cell, **predict**. The default crossover uses `k=5` retrieved chunks (a
`700`-token RAG cost) and gives a crossover around **8 chunks**. Now suppose you retrieve **more**
context per query — say `k=20` — so RAG's fixed cost rises.

**Predict:** does the crossover corpus size go **up, down, or stay the same** when RAG retrieves
more per query?

Then run it and check. *(Hint: the crossover is where stuffing first exceeds RAG's fixed cost. A
bigger `k` raises RAG's fixed cost, so stuffing has to reach a **larger** corpus before it
overtakes — the crossover moves **up**.)* The cell asserts the k=20 crossover is larger than k=5.

In [7]:
cross_k5 = cost_crossover_chunks(k=5)
cross_k20 = cost_crossover_chunks(k=20)
print('crossover with k=5  retrieved chunks:', cross_k5, 'chunks')
print('crossover with k=20 retrieved chunks:', cross_k20, 'chunks')
assert cross_k20 > cross_k5, 'retrieving more per query raises RAG cost -> crossover moves up'
print(f'\n-> retrieving more ({cross_k5} -> {cross_k20} chunks): a costlier RAG needs a bigger corpus to still win.')
print('   (The crossover is a knob: the more context RAG pulls, the later it beats stuffing.)')

crossover with k=5  retrieved chunks: 8 chunks
crossover with k=20 retrieved chunks: 23 chunks

-> retrieving more (8 -> 23 chunks): a costlier RAG needs a bigger corpus to still win.
   (The crossover is a knob: the more context RAG pulls, the later it beats stuffing.)


## What we saw

- **Cost crossover (our measurement)** — stuffing's per-query cost grows with the corpus while
  RAG's is flat; beyond a small crossover RAG is cheaper on every query, and at 100k chunks
  stuffing costs ~14,000× RAG per query.
- **Context dilution (our measurement)** — an encoder proxy: more irrelevant context shrinks the
  gold's retrieval margin (0.80 focused → ~0.19 diluted); RAG keeps it wide.
- **Lost-in-the-middle (cited, Liu et al. 2023)** — LLM accuracy dips ~20 points when the answer
  sits mid-context; retrieval places the few right chunks where the model reads best.
- **The decision** — stuff when corpus ≪ window and cost doesn't matter; retrieve when corpus ≫
  window or cost/latency matter; hybrid when you need many chunks. Corpora outgrow any window.

**Our-measurement vs cited-external:** the cost arithmetic and the encoder dilution proxy are
**our own** reproducible measurements (asserted here). The lost-in-the-middle U-curve, the RULER
effective-context gap, and the provider window sizes/prices are **cited external** constants from
their papers/docs — labelled as such, never passed off as our measurement.